### Defining Libraries and Constants

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd # Keeping it as you mentioned for potential DataFrame export

# --- Configuration Constants ---
INITIAL_PAGE_LOAD_DELAY = 3  # seconds
CONSENT_DIALOG_DELAY = 2     # seconds
FILTER_LOAD_DELAY = 3        # seconds
SCROLL_PAUSE_TIME = 2        # seconds
SCROLL_ATTEMPTS_PER_CHUNK = 2 # How many times to scroll in one go when more content is needed
MAX_WAIT_TIME = 10           # seconds for WebDriverWait

MIN_CHANNEL_LINKS = 3
MAX_CHANNEL_LINKS = 5
CHANNEL_PAGE_LOAD_DELAY = 3

SEARCH_QUERY = "tech reviews 2025" # Moved to constant

### Initialization and Closing Functions

In [2]:
def intializeWebDriver():
    """
    Initializes and returns a Chrome WebDriver instance.
    Maximizes the browser window.
    """
    print("Initializing WebDriver...")
    driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))
    driver.maximize_window()
    return driver

def closeWebDriver(driver):
    """
    Closes the WebDriver.
    """
    print("\nClosing WebDriver...")
    driver.quit()

### Helper Functions

In [3]:
def handleConsentDialog(driver, wait_time):
    """
    Attempts to click the consent/cookie dialog button if it appears.
    """
    try:
        print("Checking for consent dialog...")
        consent_button = WebDriverWait(driver, wait_time).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(@aria-label, 'Agree to the use of cookies')] | //button[contains(@aria-label, 'Reject the use of cookies')] | //button[contains(@aria-label, 'Accept all')] | //*[normalize-space(text())='Accept all'] | //*[normalize-space(text())='I agree']"))
        )
        consent_button.click()
        print("Consent dialog handled successfully.")
        time.sleep(CONSENT_DIALOG_DELAY)
    except Exception:
        print("No consent dialog found or could not click it. Proceeding...")

def navigateAndApplyVideoFilter(driver, search_query, wait_time):
    """
    Navigates to the Youtube URL and attempts to apply the 'Videos' filter.
    """
    formatted_query = search_query.replace(" ", "+")
    # Correct Youtube URL. Using googleusercontent.com is not standard for direct Youtube.
    # The standard Youtube URL is 'https://www.youtube.com/results?search_query='
    search_url = f"https://www.youtube.com/results?search_query={formatted_query}"
    print(f"Navigating to search URL: {search_url}")
    driver.get(search_url)
    time.sleep(INITIAL_PAGE_LOAD_DELAY)

    try:
        print("Attempting to apply 'Videos' filter...")
        # More robust XPath to find the "Videos" filter chip by text content
        videos_filter_xpath = "//yt-formatted-string[text()='Video' or text()='Videos']/ancestor::yt-chip-cloud-chip-renderer"
        videos_filter = WebDriverWait(driver, wait_time).until(
            EC.element_to_be_clickable((By.XPATH, videos_filter_xpath))
        )
        videos_filter.click()
        print('Successfully selected "Videos" filter.')
        time.sleep(FILTER_LOAD_DELAY)
        return True
    except Exception as e:
        print(f'Warning: Unable to click "Videos" filter: {e}. Results might include non-video content.')
        return False

def scroll_to_bottom(driver, num_scroll_attempts=1):
    """
    Scrolls the page down multiple times to load more content.
    Returns True if new content was potentially loaded, False otherwise.
    """
    last_height = driver.execute_script("return document.documentElement.scrollHeight")
    for i in range(num_scroll_attempts):
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        time.sleep(SCROLL_PAUSE_TIME) # Give time for content to load
        new_height = driver.execute_script("return document.documentElement.scrollHeight")
        if new_height == last_height:
            print(f"Scroll attempt {i+1}: No new content loaded.")
            return False # Indicate that no new content was loaded
        last_height = new_height
    print(f"Finished {num_scroll_attempts} scroll attempts.")
    return True # Indicate that some scrolling likely loaded new content

### Collecting Unique Details

In [4]:
def collectUniqueChannelLinks(driver, min_links, max_links):
    """
    Collects unique channel links from the search results page.
    Continues scrolling until min_links are found, max_links are found,
    or no more content can be loaded.
    """
    unique_channel_links = set()
    processed_video_ids = set() # To avoid reprocessing the same video elements
    scrolled_successfully_in_last_attempt = True # Assume initial page load has content

    print(f"\nAttempting to collect between {min_links} and {max_links} unique channel links.")

    while len(unique_channel_links) < max_links and \
          (len(unique_channel_links) < min_links or scrolled_successfully_in_last_attempt):

        print(f"\n--- Current unique channel links collected: {len(unique_channel_links)} ---")

        # Find all video renderers on the current page
        try:
            current_videos_on_page = WebDriverWait(driver, MAX_WAIT_TIME).until(
                EC.presence_of_all_elements_located((By.TAG_NAME, "ytd-video-renderer"))
            )
            print(f"Detected {len(current_videos_on_page)} video renderers on the page.")
        except Exception:
            print("No more 'ytd-video-renderer' elements found. Ending collection.")
            break # No videos to process, exit loop

        new_videos_processed_this_iteration = False

        for video_element in current_videos_on_page:
            if len(unique_channel_links) >= max_links:
                print(f"Reached {max_links} unique channel links. Stopping current batch processing.")
                break # Exit inner loop if max links are found

            # Get a unique ID for the video element (video URL is best)
            try:
                video_title_link_element = video_element.find_element(By.XPATH, ".//*[@id='video-title']")
                current_video_id = video_title_link_element.get_attribute("href")
            except Exception:
                current_video_id = None # Cannot get a unique ID for this element

            # Process only if it's a new video not yet processed
            if current_video_id and current_video_id not in processed_video_ids:
                processed_video_ids.add(current_video_id)
                new_videos_processed_this_iteration = True # Mark that we found a new, unprocessed video

                try:
                    # Find the channel link within the video_element
                    channel_link_element = video_element.find_element(
                        By.XPATH, ".//*[@id='channel-info']/a | .//*[@id='byline']/a"
                    )
                    channel_url = channel_link_element.get_attribute("href")

                    if channel_url:
                        unique_channel_links.add(channel_url)
                        print(f"  Added channel: {channel_url}. Total: {len(unique_channel_links)}")
                except Exception:
                    # Silently ignore if a channel link isn't found for a specific video element
                    # (e.g., ads or different video card structures)
                    pass

        # Determine if we need to scroll more for the next iteration
        if len(unique_channel_links) < max_links:
            # Scroll if:
            # 1. We found new videos in this pass (implying more content might be available below).
            # OR
            # 2. We still haven't met the minimum required links (we need to keep trying).
            if new_videos_processed_this_iteration or len(unique_channel_links) < min_links:
                print("Attempting to scroll for more content...")
                scrolled_successfully_in_last_attempt = scroll_to_bottom(driver, SCROLL_ATTEMPTS_PER_CHUNK)
                if not scrolled_successfully_in_last_attempt:
                    print("No more new content loaded after scrolling. Likely reached the end of search results.")
                    if len(unique_channel_links) < min_links:
                        print(f"WARNING: Could not find at least {min_links} channel links. Only found {len(unique_channel_links)}.")
                    break # Exit main while loop (no more content to load)
            else:
                # If no new videos were found in this iteration AND we already have >= min_links,
                # it implies we are at the end of available content and have enough channels.
                print("No new videos detected in current view, and enough channels collected (or end reached). Stopping collection.")
                break
        else:
            print(f"Reached {max_links} unique channel links. Collection complete.")

    return unique_channel_links

### Function to get Channel Details

In [5]:
def getChannelDetails(driver, channel_links, CHANNEL_PAGE_LOAD_DELAY, MAX_WAIT_TIME):
    data = []
    if channel_links:
        print('links exists')
        for channel_link in channel_links:
            print(f'Visiting channel: {channel_link}')
            try:
                driver.get(channel_link)
                time.sleep(CHANNEL_PAGE_LOAD_DELAY)
                try:
                    click_more = WebDriverWait(driver, MAX_WAIT_TIME).until(
                        EC.element_to_be_clickable((By.XPATH, '//*[@id="page-header"]/yt-page-header-renderer/yt-page-header-view-model/div/div[1]/div/yt-description-preview-view-model/truncated-text/button/span/span'))
                    )
                    click_more.click()

                    # Get nationality
                    try:
                        nationality = driver.find_element(By.XPATH, '//*[@id="additional-info-container"]/table/tbody/tr[4]/td[2]').text
                    except Exception as e:
                        print(f'No nationality found: {e}')
                    
                    # Get joined Date
                    try:
                        joinedOn = driver.find_element(By.XPATH, '//*[@id="additional-info-container"]/table/tbody/tr[5]/td[2]/yt-attributed-string/span/span').text
                    except:
                        print(f'No joined_date found: {e}')
                    
                    # Get subscribers
                    try:
                        subscribers = driver.find_element(By.XPATH, '//*[@id="additional-info-container"]/table/tbody/tr[6]/td[2]').text
                    except Exception as e:
                        print(f'Unable to get subscribers: {e}')
                    
                    # get videosCount
                    try:
                        videosCount = driver.find_element(By.XPATH, '//*[@id="additional-info-container"]/table/tbody/tr[7]/td[2]').text
                    except Exception as e:
                        print(f'Unable to get video Count: {e}')

                    # get Views
                    try:
                        totalViews = driver.find_element(By.XPATH, '//*[@id="additional-info-container"]/table/tbody/tr[8]/td[2]').text
                    except Exception as e:
                        print(f'Unable to get total Views: {e}')
                    
                    try:
                        data.append({
                            'nationality': nationality,
                            'joined_on': joinedOn,
                            'subscribers': subscribers,
                            'videos_count': videosCount,
                            'total_views': totalViews,
                        })
                    except Exception as e:
                        print(f'Unable to add Data: {e}')
                except Exception as e:
                    print(f'Unable to click `Click More`: {e}')
            except Exception as e:
                print(f'Unable to fetch channel url: {e}')
    return data
            


### Main Function

In [6]:
if __name__ == '__main__':
    driver = intializeWebDriver()
    try:
        # Step 1: Handle consent dialog (if it appears)
        handleConsentDialog(driver, MAX_WAIT_TIME)

        # Step 2: Navigate to search results and apply video filter
        navigateAndApplyVideoFilter(driver, SEARCH_QUERY, MAX_WAIT_TIME)

        # Step 3: Collect the unique channel links
        channelLinks = collectUniqueChannelLinks(driver, MIN_CHANNEL_LINKS, MAX_CHANNEL_LINKS)

        # Step 4: Final output
        print(channelLinks)
        records = pd.DataFrame(channelLinks, columns=['channel_url'])
        print(records)
        # records.to_csv('channel_urls.csv', index=False)
        details = getChannelDetails(driver, channelLinks, CHANNEL_PAGE_LOAD_DELAY, MAX_WAIT_TIME)
        recordDetails = pd.DataFrame(data=details)
        print(recordDetails)

    finally:
        # Ensure the WebDriver is closed even if an error occurs
        closeWebDriver(driver)

Initializing WebDriver...
Checking for consent dialog...
No consent dialog found or could not click it. Proceeding...
Navigating to search URL: https://www.youtube.com/results?search_query=tech+reviews+2025
Attempting to apply 'Videos' filter...
Successfully selected "Videos" filter.

Attempting to collect between 3 and 5 unique channel links.

--- Current unique channel links collected: 0 ---
Detected 20 video renderers on the page.
  Added channel: https://www.youtube.com/@PCMag. Total: 1
  Added channel: https://www.youtube.com/@JarrodsTech. Total: 2
  Added channel: https://www.youtube.com/@TrakinTech. Total: 3
  Added channel: https://www.youtube.com/@ReneRitchie. Total: 4
  Added channel: https://www.youtube.com/@TechMagnet. Total: 5
Reached 5 unique channel links. Stopping current batch processing.
Reached 5 unique channel links. Collection complete.
{'https://www.youtube.com/@PCMag', 'https://www.youtube.com/@ReneRitchie', 'https://www.youtube.com/@TechMagnet', 'https://www.you